In [1]:
from collections import Counter, defaultdict
import json
import pandas as pd
from pathlib import Path

In [53]:
dataset_path = Path("/Users/fangzhouma/Desktop/3d_vision/3D-VLM-benchmark-OOS/outputs/jsonl_v2/multi_turn.jsonl")
# Change this path if your JSONL is elsewhere.

raw_docs = []
with dataset_path.open("r") as f:
    for line in f:
        raw_docs.append(json.loads(line))

len(raw_docs), raw_docs[0].keys()

(30,
 dict_keys(['trajectory_id', 'question_class', 'video_id', 'video_path', 'query_time_sec', 'query_time_in_clip_sec', 'clip_start_time_sec', 'clip_end_time_sec', 'clip_duration_sec', 'horizon_sec', 'object_a_assoc_id', 'object_a_name', 'num_incremental_steps', 'num_branch_steps', 'terminated_at_step', 'stop_reason', 'generation_info', 'doc_id', 'mode', 'steps', 'include_gold_history', 'gold_history_messages']))

In [54]:
rows = []

for raw in raw_docs:
    for step in raw.get("steps", []):
        if step.get("skipped"):
            continue

        rows.append({
            "trajectory_id": raw.get("trajectory_id"),
            "video_id": raw.get("video_id"),
            "doc_id": raw.get("doc_id"),
            "mode": raw.get("mode"),
            "query_time_sec": raw.get("query_time_sec"),
            "step": str(step.get("step")),
            "branch_group": step.get("branch_group"),
            "depends_on_steps": step.get("depends_on_steps"),
            "question_class": step.get("step_question_class"),
            "question": step.get("question"),
            "choices": step.get("choices") or [],
            "correct_idx": step.get("correct_idx"),
            "target_text": step.get("target_text"),
            "acceptable_idxs": step.get("acceptable_idxs"),
            "answer_metadata": step.get("answer_metadata"),
        })

df = pd.DataFrame(rows)
df.head()

,trajectory_id,video_id,doc_id,mode,query_time_sec,step,branch_group,depends_on_steps,question_class,question,choices,correct_idx,target_text,acceptable_idxs,answer_metadata
0,P01-20240203-132119__oos_staged_h5p0_0,P01-20240203-132119,P01-20240203-132119__oos_staged_h5p0_0,multi_turn,152.0,1,NaN,[],oos_step1_visibility,"At the current time <TIME 00:02:32.0 video 1>,...","[No, Yes]",0.0,No,None,"{'status': 'out_of_view', 'is_visible': False,..."
1,P01-20240203-132119__oos_staged_h5p0_0,P01-20240203-132119,P01-20240203-132119__oos_staged_h5p0_0,multi_turn,152.0,2,NaN,[1],oos_step2_last_visible,When was the previously moved empty mug last v...,[],NaN,"<TIME 00:02:26.0 video 1>; Point=(0.4974, 0.7829)",None,"{'sampled_last_visible_time_sec': 146.0, 'samp..."
2,P01-20240203-132119__oos_staged_h5p0_0,P01-20240203-132119,P01-20240203-132119__oos_staged_h5p0_0,multi_turn,152.0,3,NaN,"[1, 2]",oos_step3_last_placement,At what time did the previously moved empty mu...,[],NaN,"<TIME 00:02:18.4 video 1>; Point=(0.6612, 0.7427)",None,"{'last_placement_time_sec': 138.4, 'last_place..."
3,P01-20240203-132119__oos_staged_h5p0_0,P01-20240203-132119,P01-20240203-132119__oos_staged_h5p0_0,multi_turn,152.0,4,NaN,"[1, 2, 3]",oos_step4_fixture,"At the current time <TIME 00:02:32.0 video 1>,...","[shelf, fridgefreezer, counter, bin, drawer]",2.0,counter,None,"{'reference_time_sec': 138.4, 'correct_fixture..."
4,P01-20240203-132119__oos_staged_h5p0_0,P01-20240203-132119,P01-20240203-132119__oos_staged_h5p0_0,multi_turn,152.0,5a,post_step4,"[1, 2, 3, 4]",oos_branch_object_camera_relative_position,"At the current time <TIME 00:02:32.0 video 1>,...","[Front-right, Back-right, Front-left, Back-left]",1.0,Back-right,None,"{'reference_time_sec': 152.0, 'camera_coordina..."


In [55]:
for qclass, g in df.groupby("question_class"):
    print("\n" + "=" * 100)
    print(qclass)
    print("N:", len(g))

    print("\nTarget text distribution:")
    print(g["target_text"].value_counts(dropna=False))

    print("\nCorrect idx distribution:")
    print(g["correct_idx"].value_counts(dropna=False))

    print("\nChoice order distribution:")
    print(g["choices"].apply(tuple).value_counts().head(10))


oos_branch_counter_area
N: 30

Target text distribution:
target_text
beside the sink, and below the boiler    17
beside the hob and near the sink          8
close to the microwave                    3
between the fridge and the hob            1
the narrow stripe area along the sink     1
Name: count, dtype: int64

Correct idx distribution:
correct_idx
0.0    8
1.0    7
4.0    6
2.0    6
3.0    3
Name: count, dtype: int64

Choice order distribution:
choices
(between the fridge and the hob, hob-side corner, next to the sink, close to the microwave, beside the hob and near the sink, beside the sink, and below the boiler)                   1
(close to the microwave, between the fridge and the hob, beside the sink, and below the boiler, beside the hob and near the sink, the narrow stripe area along the sink)               1
(beside the hob and near the sink, hob-side corner, next to the sink, between the fridge and the hob, close to the microwave, the narrow stripe area along the sink)    

In [56]:
baseline_rows = []

for qclass, g in df.groupby("question_class"):
    n = len(g)

    semantic_counts = g["target_text"].value_counts(dropna=False)
    idx_counts = g["correct_idx"].value_counts(dropna=False)

    semantic_majority_acc = semantic_counts.iloc[0] / n
    idx_majority_acc = idx_counts.iloc[0] / n

    baseline_rows.append({
        "question_class": qclass,
        "n": n,
        "semantic_majority": semantic_counts.index[0],
        "semantic_majority_acc": semantic_majority_acc,
        "idx_majority": idx_counts.index[0],
        "idx_majority_acc": idx_majority_acc,
    })

baseline_df = pd.DataFrame(baseline_rows).sort_values("question_class")
baseline_df




,question_class,n,semantic_majority,semantic_majority_acc,idx_majority,idx_majority_acc
0,oos_branch_counter_area,30,"beside the sink, and below the boiler",0.566667,0.0,0.266667
1,oos_branch_object_camera_relative_position,30,Back-right,0.600000,1.0,0.300000
2,oos_branch_object_object_distance,30,medium,0.433333,2.0,0.366667
3,oos_branch_object_object_relation,30,Back-right,0.633333,2.0,0.366667
4,oos_step1_visibility,30,No,1.000000,1.0,0.600000
5,oos_step2_last_visible,30,"<TIME 00:02:26.0 video 1>; Point=(0.4974, 0.7829)",0.033333,NaN,1.000000
6,oos_step3_last_placement,30,"<TIME 00:02:18.4 video 1>; Point=(0.6612, 0.7427)",0.233333,NaN,1.000000
7,oos_step4_fixture,30,counter,1.000000,2.0,0.266667


In [57]:
step1 = df[df["question_class"] == "oos_step1_visibility"].copy()

step1["choices_tuple"] = step1["choices"].apply(tuple)

step1[["choices_tuple", "correct_idx", "target_text"]].value_counts()

choices_tuple  correct_idx  target_text
(Yes, No)      1.0          No             18
(No, Yes)      0.0          No             12
Name: count, dtype: int64

In [47]:
dist = df[df["question_class"] == "oos_branch_object_object_distance"]

print(dist["choices"].apply(len).value_counts())
print(dist["correct_idx"].value_counts(dropna=False))

bad = dist[
    (dist["choices"].apply(len) != 3)
    | (~dist["correct_idx"].isin([0, 1, 2]))
]

bad[["trajectory_id", "step", "target_text", "choices", "correct_idx"]]

choices
3    30
Name: count, dtype: int64
correct_idx
1.0    11
2.0    10
0.0     9
Name: count, dtype: int64


,trajectory_id,step,target_text,choices,correct_idx
